# GaBP State Estimation in Gas Networks - Demirel et al.

Implementation of the GaBP state estimator from the Demirel paper. The state of the system is given as a vector of node pressures $\mathbf{x} = [p_1, \dots, p_n]^T$ in eq. 16. Measurements are then collated via the loopy GaBP algorithm and factor graphs.

In [20]:
import numpy as np
from scipy import sparse
from pygsp import graphs
import math
import matplotlib.pyplot as plt
from typing import NamedTuple


**Initialisation of factors and building factor graph**

Each measurement becomes a factor node $f_i$ carrying a linearised local function
$\psi_i(\mathcal{V}_i) \propto \exp\!\big(-\tfrac{1}{2}[z_i - h_i(x)]^2 / \sigma_{z_i}^2\big)$
(eqs. 5, 11). A factor stores the variables it touches, the Jacobian coefficients
$C_{x_s} = \partial h_i / \partial x_s$ (eq. 13), the measurement $z_i$, and its
variance $\sigma_{z_i}^2$. `var_factors[s]` is the adjacency list of factors
incident to variable $s$. Virtual factors with large variance ($\sigma^2 = 1/\epsilon$)
encode "no prior information" at unmeasured nodes.

In [62]:
class Factor(NamedTuple):
    vars: list[int]     # the node indices this factor touches
    C: list[float]      # coefficient on each, aligned with vars
    z: float            # measurement
    var_z: float        # measurement variance


# This is the GMRF factor graph that encodes a smooth prior i.e neighbouring nodes have similar pressure.
# Estimator 2 from initial notebook + GaBP
def build_factor_graph(y, sigma, edges, gamma):
    N = len(y) # placeholder
    factors = []
    for i in range(N):
        factors.append(Factor([i], [1], y[i], sigma[i]))

    for i, j, w in edges:
        factors.append(Factor([i, j], [1, -1], 0, 1/(gamma*w)))

    eps = 1e-4
    for i in range(N):
        factors.append(Factor([i], [1.0], 0.0, 1/eps))
    
    return factors, N

# This is the factor graph that encodes the fluid physics from below cells i.e not a smooth prior
def build_demirel_factor_graph(pressure_meas, flow_meas, injection_meas, x_OP, edges, neighbours, N):
    factors = []

    # Extracting R_ij from edges list of tuples 
    # (TODO: Watch for lookup mistakes when flow is keyed differently)
    R_of = {(i, j): R for (i, j, R) in edges}

    # Pressure at node i 
    for i, (z, var_z) in pressure_meas.items():
        factors.append(Factor([i], [1.0], z, var_z))

    # Volume flow at edge
    for (i, j), (z, var_z) in flow_meas.items():
        
        R = R_of[(i, j)]
        vars = [i, j]
        
        c = coeffs(x_OP[i], x_OP[j], R)
        C = [c, -c]
        C_dot_xOP = sum(cc * x_OP[v] for cc, v in zip(C, vars))
        z_eff = z - h_flow(x_OP[i], x_OP[j], R) + C_dot_xOP

        factors.append(Factor(vars, C, z_eff, var_z))

    # Volume flow at injection/extraction 
    for i, (z, var_z) in injection_meas.items():
        vars = [i]
        C = [0.0]

        for (k, R_ik) in neighbours[i]:
            c = coeffs(x_OP[i], x_OP[k], R_ik)
            vars.append(k)
            C.append(c)
            C[0] -= c

        C_dot_xOP = sum(cc * x_OP[v] for cc, v in zip(C, vars))
        z_eff = z - h_injection(i, x_OP , neighbours) + C_dot_xOP
        factors.append(Factor(vars, C, z_eff, var_z))

    var_factors = index_factors(factors, N)
    return factors, var_factors

# Avoids redundant code from build_factor_graph() & build_demirel_factor_graph() 
def index_factors(factors, N):
    var_factors = [[] for _ in range(N)]
    for f_id, f in enumerate(factors):
        for s in f.vars:
            var_factors[s].append(f_id)
    return var_factors

"""
# edge table from worked example with weights edge = (node1, node2, weight)
edges = [
    (0, 1, 1.5),
    (1, 2, 2),
    (2, 3, 1.8),
    (3, 4, 1.2),
    (1, 4, 0.8),
    (4, 5, 2.1),
    (5, 6, 1.6)
]
"""

sigma = np.array([0.25, 0.25, 0.25, 225, 0.25, 0.25, 0.25])             # drift injection - variances  with additional drift at node 4.
y = np.array([120.15, 114.48, 108.38, 108.47, 109.02, 94.35, 85.06])    # measurements with drift injection - N(0, 0.25)
gamma = 0.3

pressure_meas  = {i: (p[i], 0.25) for i in range(N)}
flow_meas      = {(i, j): (h_flow(p[i], p[j], R), 0.25) for (i, j, R) in gas_edges}
injection_meas = {1: (h_injection(1, p, neighbours), 0.25)}
rng = np.random.default_rng(0)
readings = p + rng.normal(0, 0.05, size=N)

factors, var_factors = build_demirel_factor_graph(
    pressure_meas, flow_meas, injection_meas, readings, gas_edges, neighbours, N
)

N = len(y)
Lambda = np.zeros((N,N))
b = np.zeros(N)
for f in factors:
    c = np.zeros(N)
    c[f.vars] = f.C
    Lambda += np.outer(c, c) / f.var_z
    b += c * f.z / f.var_z
print(np.linalg.solve(Lambda, b))
test = run_gabp(factors, var_factors, N, max_iters=200, tol=1e-8, beta=0.5)
print(test)
#for f in factors:
#    if len(f.vars) == 2:
#        #print(f.vars, f.C)




[1.00973327 1.00292752 0.94933399 0.95032328 0.99454049 0.9618626
 0.89127886]


UnboundLocalError: cannot access local variable 'damp' where it is not associated with a value

**GaBP Algorithm**

Loopy GaBP is implemented with the sum-product algo (Pearl 1988) and Gaussian messages. 

**Synchronous flooding schedule:** 
Two message types in canonical form, $(\Lambda, \eta) = (1/\sigma^2,\ \mu/\sigma^2)$:

- **Variable → factor** (eqs. 6, 8): product of all *other* incoming
  factor messages — a sum in canonical form,
  $\frac{1}{\sigma^2_{x_s \to f_i}} = \sum_{f_a \in \mathcal{F}_s \setminus f_i} \frac{1}{\sigma^2_{f_a \to x_s}}$.
- **Factor → variable** (eqs. 10, 12): solve the linearised measurement for $x_s$,
  $z_{f_i \to x_s} = \frac{1}{C_{x_s}}\big(z_i - \sum_{b \neq s} C_{x_b} z_{x_b \to f_i}\big)$.

Belief at each node (eqs. 14, 15): product of all incident factor messages,
$\hat{x}_s = \big(\sum_{f_c \in \mathcal{F}_s} z_{f_c \to x_s}/\sigma^2_{f_c \to x_s}\big)\,\hat{\sigma}^2_{x_s}$.

In [61]:
"""
factors     : the list of factors from NamedTuple class - Factor(vars, C, z, var_z)
var_factors : var_factors[s] = the list of factors with id (f_id) touching variable s
v2f, f2v    : variable-to-factor & factor-to-variable respectively
"""

def links(factors):
    """Function that produces a lazy generator object as and when we need it in loops"""
    for f_id, f in enumerate(factors):
        for s in f.vars:
            yield (f_id, s)

def init_messages(factors):
    """Initilaises uninformative prior for messages i.e (Lam, eta) = (0, 0)"""
    return {(f_id, s): (0.0, 0.0) for f_id, s in links(factors)}

def var_to_factor(f_id, s, f2v, var_factors):
    """Variable to factor message: just a sum over the other factors on s"""
    Lam = eta = 0.0
    for fp in var_factors[s]:
        if fp == f_id: # (excludes the factor we are sending the message to)
            continue
        Lf, ef = f2v[(fp, s)] # (f -> v incoming messages)
        Lam += Lf
        eta += ef
    return (Lam, eta)

def factor_to_var(f_id, s, v2f, factors):
    f = factors[f_id]
    Cs = f.C[f.vars.index(s)]
    acc_mean = acc_var = 0.0
    for b, Cb in zip(f.vars, f.C):
        if b == s:
            continue
        Lb, eb = v2f[(f_id, b)] # (v -> f incoming messages)
        if Lb == 0.0:
            return (0.0, 0.0)
        mu_b, var_b = eb / Lb, 1.0 / Lb
        acc_mean += Cb * mu_b
        acc_var += Cb **2 * var_b
    mean = (f.z - acc_mean) / Cs
    var = (f.var_z + acc_var) / Cs**2
    Lam = 1.0 / var
    return (Lam, mean*Lam)

def belief(s, f2v, var_factors):
    """Our belief at s which is just a sum over factors on s"""
    Lam = eta = 0.0
    for fp in var_factors[s]:
        Lf, ef = f2v[(fp,s)]
        Lam += Lf
        eta += ef
    return (eta / Lam, 1.0 / Lam) # (just (mean, variance))

def run_gabp(factors, var_factors, N, max_iters=200, tol=1e-8, beta=0.5):
    """Synchronous flooding"""
    v2f = init_messages(factors)
    f2v = init_messages(factors)
    prev = None
    for _ in range(max_iters):
        v2f_raw = {(f_id, s): var_to_factor(f_id, s, f2v, var_factors) for f_id, s in links(factors)}
        v2f = {k: damp(v2f_raw[k], v2f[k], beta) for k in v2f_raw}
        f2v_raw = {(f_id, s): factor_to_var(f_id, s, v2f, factors) for f_id, s in links(factors)}
        f2v = {k: damp(f2v_raw[k], f2v[k], beta) for k in f2v_raw}
        cur = {s: belief(s, f2v, var_factors) for s in range(N)}
        #print(v2f, f2v, cur)

        def damp(new, old, beta):
            Ln, en = new
            Lo, eo = old
            return ((1 - beta) * Ln + beta*Lo, (1 - beta)*en + beta*eo)
        
        def converged(cur, prev, tol):
            deltas = []
            for s in cur:
                mean, var = cur[s]
                mean_prev, var_prev = prev[s]
                deltas.append(abs(mean - mean_prev))
                deltas.append(abs(var - var_prev))
            return max(deltas) < tol

        if prev is not None and converged(cur, prev, tol):
            break
        prev = cur
    return cur

#cur = run_gabp(factors, var_factors, N, max_iters=20, tol=1e-8)
#print(cur)
#print(len(factors))


**Darcy-Weisbach equations & Jacobian implementation**

Pressure drop along a pipe follows Darcy–Weisbach,
$|\Delta p_{ij}| = R_{ij}\,\dot V_{ij}^2$ (eq. 17), giving the three measurement
functions: node pressure $h_{p_i} = p_i$ (eq. 20), edge flow
$h_{\dot V_{ij}} = \operatorname{sgn}(p_i - p_j)\sqrt{|p_i - p_j|/R_{ij}}$ (eq. 21),
and nodal injection $h_{\dot V_i} = -\sum_{k \in Nb(i)} h_{\dot V_{ik}}$ (eq. 22).

Because the flow functions are nonlinear, GaBP requires the linearised Jacobian
$C_{x_s} = \partial h_i / \partial x_s$ (eq. 13), evaluated at an operating point
$x_{OP}$ from an initial pipe-flow solve (eq. 23). The edge-flow sensitivity is
$\partial \dot V_{ij}/\partial p_i = +\tfrac{1}{2}\big(\sqrt{R_{ij}}\sqrt{|u|}\big)^{-1}$,
$\partial/\partial p_j$ equal and opposite, with $u = p_i - p_j$ floored at
$\epsilon$ to guard the $|u|^{-1/2}$ singularity at zero flow.


In [23]:
# (i, j, R_ij)
gas_edges = [
    (0, 1, 0.5),
    (1, 2, 1.2),
    (2, 3, 0.8),
    (3, 4, 2.0),
    (1, 4, 1.5),
    (4, 5, 0.6),
    (5, 6, 1.0),
]

p = np.array([1.00, 0.99, 0.97, 0.96, 0.98, 0.94, 0.92])
N = len(p)

neighbours = {i: [] for i in range(N)}
for i, j, R in gas_edges:
    neighbours[i].append((j, R))
    neighbours[j].append((i, R))

def h_flow(p_i, p_j, R_ij):
    return np.sign(p_i - p_j) * np.sqrt(abs(p_i - p_j) / R_ij)

def h_injection(i, p, neighbours):
    total = 0.0
    for k, R_ik in neighbours[i]:
        total += h_flow(p[i], p[k], R_ik)
    return -total

def coeffs(p_i, p_j, R_ij, eps=1e-4):
    u = p_i - p_j
    c = 0.5 / (np.sqrt(R_ij)*np.sqrt(max(abs(u), eps)))
    return c

def jacobian(p, edges, neighbours, pressure_meas_nodes, injection_meas_nodes):
    rows = []
    # `edge_c = {(i, j): coeffs(p[i], p[j], R) for (i, j, R) in edges}' - can do this if having computation issues but 
    # at current scale easier and readable to just call coeffs for each loop

    # pressure rows or eq. 20
    for i in pressure_meas_nodes:
        row = np.zeros(N)
        row[i] = 1.0
        rows.append(row)

    # edge-flow rows or eq. 21
    for (i, j, R) in edges:
        c = coeffs(p[i], p[j], R)
        row = np.zeros(N)
        row[i] = +c
        row[j] = -c
        rows.append(row)

    # injection rows or eq. 22
    for i in injection_meas_nodes:
        row = np.zeros(N)
        for (k, R_ik) in neighbours[i]:
            c = coeffs(p[i], p[k], R_ik)
            row[i] -= c
            row[k] += c
        rows.append(row)
    
    return np.stack(rows)

J = jacobian(p, gas_edges, neighbours, pressure_meas_nodes=range(N), injection_meas_nodes=[1])
